<a href="https://colab.research.google.com/github/chelseanbr/aih-final-project/blob/main/notebooks/04-evaluate-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluate Models

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
test_data = [json.loads(line) for line in open("/content/drive/MyDrive/final_mcqs_test.jsonl")]

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re
from tqdm import tqdm

def evaluate_model_batched(model_name_or_path, test_data, subset_size=50, batch_size=4, model_description=None):
    print(f"\nEvaluating model: {model_description or model_name_or_path}")

    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(model_name_or_path, device_map="auto", torch_dtype=torch.float16)
    model.eval()

    # Slice test data
    data = test_data[:subset_size]

    correct = 0
    total = 0
    num_a = num_b = num_c = num_d = num_skipped = 0

    # Process in batches
    for i in tqdm(range(0, len(data), batch_size)):
        batch = data[i:i + batch_size]
        prompts = [x["prompt"].strip() for x in batch]
        true_answers = [re.search(r"<answer>([A-D])</answer>", x["completion"]) for x in batch]
        true_answers = [m.group(1) if m else None for m in true_answers]

        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for j, decoded in enumerate(decoded_outputs):
            prompt = prompts[j]
            true_answer = true_answers[j]
            generated = decoded[len(prompt):].strip()

            pred_matches = re.findall(r"<answer>([A-D])</answer>", generated)
            if len(pred_matches) != 1 or not true_answer:
                num_skipped += 1
                continue

            pred_answer = pred_matches[0]
            if pred_answer == "A": num_a += 1
            elif pred_answer == "B": num_b += 1
            elif pred_answer == "C": num_c += 1
            elif pred_answer == "D": num_d += 1

            if pred_answer == true_answer:
                correct += 1
            total += 1

    accuracy = correct / total if total > 0 else 0

    print(f"\nAccuracy on {total} valid samples: {accuracy:.2%} ({correct}/{total})")
    print(f"Skipped due to multiple/no <answer>: {num_skipped}")
    print("Answer Distribution:")
    print(f"  A: {num_a}")
    print(f"  B: {num_b}")
    print(f"  C: {num_c}")
    print(f"  D: {num_d}")

## Fine-Tuned TinyLLama

In [ ]:
model_path = "/content/drive/MyDrive/tinyllama-mcq-lora"
evaluate_model_batched(model_path, test_data, subset_size=len(test_data), batch_size=4, model_description='Finetuned TinyLlama')

Using device: cuda


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

['What are the treatments for Infantile Refsum Disease?\nA. Avoidance of foods rich in phytanic acid\nB. Antioxidant therapy\nC. Liver transplantation\nD. Enzyme replacement therapy']


Evaluating: 100%|██████████| 1/1 [00:12<00:00, 12.02s/it]


There is no standard course of treatment for Infantile Refsum Disease. Treatment is symptomatic and supportive.
<answer>B</answer>
<answer>C</answer>
<answer>D</answer>
<answer>A</answer>
<answer>B</answer>
<answer>D</answer>
<answer>A</answer>
<answer>C</answer>
<answer>B</answer>
<answer>D</answer>
<answer>A</answer>
<answer>C</answer>
<answer>D</answer>
<answer>B</answer>
<answer>D</answer>
<answer>A</answer>
<answer>C</answer>
<answer>B</answer>
<answer>D</answer>
<answer>A</answer>
<answer>C</answer>
<answer>D</answer>
<answer>B</answer>
<answer>D</answer>
<answer>A</answer>
<answer>C</answer>
<answer>B</answer>
<answer>D</answer>
<answer

Accuracy: 0.00% (0/1)
Number of As: 0
Number of Bs: 1
Number of Cs: 0
Number of Ds: 0


## Baseline TinyLLama

In [ ]:
model_path = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
evaluate_model_batched(model_path, test_data, subset_size=len(test_data), batch_size=4, model_description='Baseline TinyLlama')

Using device: cuda


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

['What are the treatments for Infantile Refsum Disease?\nA. Avoidance of foods rich in phytanic acid\nB. Antioxidant therapy\nC. Liver transplantation\nD. Enzyme replacement therapy']


Evaluating: 100%|██████████| 1/1 [00:06<00:00,  6.27s/it]


E. None of the above

Questions 10-12:

10. Which of the following is not a treatment for Infantile Refsum Disease?
A. Antioxidant therapy
B. Enzyme replacement therapy
C. Liver transplantation
D. None of the above

Questions 13-15:

13. Which of the following is a potential cause of Infantile Refsum Disease?
A. Vitamin A deficiency
B. Vitamin D deficiency
C. Vitamin E deficiency
D. Vitamin B12 deficiency

Questions 16-18:

16. Which of the following is a potential treatment for Infantile Refsum Disease?
A. Antioxidant therapy
B. Enzyme replacement therapy
C. Liver transplantation
D. None of the above

Questions 19-21:

19. Which of the following is a potential cause of Infantile Refsum Disease?
A. Vitamin A deficiency

Accuracy: 0.00% (0/1)
Number of As: 0
Number of Bs: 0
Number of Cs: 0
Number of Ds: 0


## MedAlpaca

In [ ]:
model_path "medalpaca/medalpaca-7b") 
evaluate_model_batched(model_path, test_data, subset_size=len(test_data), batch_size=4, model_description='MedAlpaca 7B')

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/7.18G [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:609: UserWarning: `pad_token_id` should be positive but got -1. This will cause errors when batch generating, if there is padding. Please set `pad_token_id` explicitly as `model.generation_config.pad_token_id=PAD_TOKEN_ID` to avoid errors in generation
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 8.88 MiB is free. Process 16652 has 39.54 GiB memory in use. Of the allocated memory 38.97 GiB is allocated by PyTorch, and 79.24 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)